# RinFix — Entrenamiento de YOLO para rines

Este cuaderno entrena **dos modelos pequeños** que después corren en el teléfono con LiteRT:

1. **Modelo de rin** (`rin`): encuentra el rin en la foto para recortarlo.
2. **Modelo de daños** (`danos`): busca daños visibles dentro del recorte del rin (rayón, pintura, golpe, corrosión, fisura, según los datos que haya).

Se entrenan por separado porque los datasets públicos están etiquetados a medias: los de rines no marcan daños y los de daños no marcan el rin. Mezclarlos le enseñaría al modelo cosas falsas.

**Antes de ejecutar**

1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.
2. Crea una cuenta gratis en [roboflow.com](https://roboflow.com) y copia tu clave privada de API (`Settings → API Keys`).
3. En Colab, abre el panel de la llave 🔑 (Secretos), crea un secreto llamado `ROBOFLOW_API_KEY` con tu clave y activa el acceso para este cuaderno.
4. `Entorno de ejecución → Ejecutar todo`. Cuando pida permiso para Google Drive, acéptalo: ahí se guardan los modelos.

Tiempo aproximado con T4: 30 a 60 minutos.

**Importante:** los resultados de este cuaderno son una primera medición con datos públicos. No reemplazan la prueba con fotos reales de rines tomadas como lo hará un cliente.

## 1. Revisar la GPU e instalar librerías

In [ ]:
!nvidia-smi -L || echo "No hay GPU: cambia el tipo de entorno a GPU T4 antes de entrenar."

In [ ]:
%pip install -q -U ultralytics roboflow

import ultralytics
ultralytics.checks()

## 2. Configuración

In [ ]:
from datetime import datetime
from pathlib import Path

# Carpeta de trabajo dentro de Colab (se borra al cerrar la sesión)
TRABAJO = Path("/content/rinfix")
RAW_DIR = TRABAJO / "descargas"
DATA_DIR = TRABAJO / "datos"
RUNS_DIR = TRABAJO / "runs"
for carpeta in (RAW_DIR, DATA_DIR, RUNS_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

# Parámetros de entrenamiento
IMGSZ = 640                      # tamaño de entrada; el mismo que usará la app
BATCH = 16
MODELO_BASE_RIN = "yolov8n.pt"   # 'n' es el más liviano para el teléfono
MODELO_BASE_DANOS = "yolov8n.pt" # prueba "yolov8s.pt" si quieres más precisión a costa de velocidad
EPOCAS_RIN = 50
EPOCAS_DANOS = 120
PACIENCIA_RIN = 15               # se detiene si no mejora en este número de épocas
PACIENCIA_DANOS = 30
MIN_CAJAS = 30                   # una clase con menos cajas de entrenamiento se descarta

# Salida en Google Drive
from google.colab import drive
try:
    drive.mount("/content/drive")
    SALIDA = Path("/content/drive/MyDrive/RinFix/ml/modelos") / datetime.now().strftime("%Y%m%d-%H%M")
except Exception as error:
    print("No se montó Drive, se guardará en /content/salida:", error)
    SALIDA = Path("/content/salida") / datetime.now().strftime("%Y%m%d-%H%M")
SALIDA.mkdir(parents=True, exist_ok=True)
print("Los modelos se guardarán en:", SALIDA)

In [ ]:
# Clave de Roboflow: se lee de los Secretos de Colab y nunca se imprime ni se guarda en el cuaderno.
from getpass import getpass

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    print("Clave de Roboflow leída desde Secretos.")
except Exception:
    ROBOFLOW_API_KEY = getpass("Pega tu clave privada de Roboflow: ")

## 3. Fuentes de datos y reglas de clases

In [ ]:
# Datasets públicos de Roboflow Universe (todos con licencia CC BY 4.0 según su página).
# tipo "rin"   -> sirven para ubicar el rin en la foto.
# tipo "danos" -> fotos de rines con daños marcados.
# incluir=False -> se descarga y se muestra, pero no entra al entrenamiento hasta revisarlo.
# solo=[...]     -> de ese dataset solo se usan las clases cuyo nombre contenga alguna de esas palabras.
FUENTES = [
    {"workspace": "car-makemodel", "proyecto": "wheel-rims-detection", "tipo": "rin", "incluir": True,
     "nota": "Rines de carro, clase Wheel-Rim"},
    {"workspace": "mv-n5n8l", "proyecto": "rim-dataset", "tipo": "rin", "incluir": True,
     "nota": "Una sola clase: rim"},
    {"workspace": "tim-4ijf0", "proyecto": "alloy-wheel-defect-4fkug", "tipo": "danos", "incluir": True,
     "nota": "Rines de aleación: scratch y paint-damage"},
    {"workspace": "cmtscanner-3crf0", "proyecto": "rim-scratches", "tipo": "danos", "incluir": True,
     "nota": "Rayones en rines. Es de segmentación: se descarga en COCO y se convierte a cajas"},
    {"workspace": "allen-frederick-xwul1", "proyecto": "clearquote-you9t", "tipo": "danos", "incluir": True,
     "solo": ["alloy"],
     "nota": "Inspección de carros. Solo se usan las clases de rin (alloy curbrash, alloy bent); "
             "las de carrocería se ignoran"},
]
# Descartados después de la primera prueba:
# - project-5cewq/rim-scratches-arjkv: no tiene versiones publicadas.
# - taufiq-6bxh7/rim_defect: privado o eliminado.
# - fall-detection-sgbit/wheel-rim-defect-detection-w62ez: defectos de soldadura de fábrica, no daños de uso.

# Reglas para pasar los nombres originales a las clases de RinFix (se comparan en minúsculas).
REGLAS_RIN = {"rin": ["rim", "wheel", "llanta", "rin"]}
EXCLUIR_RIN = ["tire", "tyre", "neumatico", "damage", "defect", "scratch", "crack", "dent"]

REGLAS_DANOS = {
    "fisura": ["crack", "fisura", "grieta"],
    "golpe": ["dent", "bent", "bend", "deform", "impact", "golpe"],
    "corrosion": ["corros", "rust", "oxid"],
    "rayon": ["scratch", "scrape", "curb", "rash", "abras", "rayon", "rayado"],
    "pintura": ["paint", "peel", "chip", "flak", "coating", "pintura"],
}
EXCLUIR_DANOS = ["tire", "tyre", "discolor", "shelling"]
ORDEN_DANOS = ["rayon", "pintura", "golpe", "corrosion", "fisura"]

# Si después de ver las muestras un nombre quedó mal, se corrige aquí. Ejemplo:
# MAPEO_MANUAL = {"cmtscanner-3crf0/rim-scratches": {"object": "rin"}}
# Usa None como valor para ignorar una clase.
MAPEO_MANUAL = {}

print(f"{len(FUENTES)} fuentes configuradas.")

In [ ]:
# Funciones de apoyo: lectura de etiquetas, mapeo de clases, unión de datasets y vistas previas.
import hashlib
import json
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
from PIL import Image, ImageDraw

EXT_IMAGEN = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLITS = ["train", "valid", "test"]
COLORES = ["#2E9E5B", "#E0A800", "#5B7BD5", "#C8341A", "#8E44AD", "#16A085", "#D35400"]


def normalizar(nombre):
    """'Paint-damage' -> 'paint damage'."""
    return re.sub(r"[\s_\-]+", " ", str(nombre).strip().lower())


def mapear_clase(nombre, reglas, excluir=(), manual=None):
    """Devuelve la clase de RinFix para un nombre original, o None si se ignora."""
    if manual and nombre in manual:
        return manual[nombre]
    n = normalizar(nombre)
    if any(palabra in n for palabra in excluir):
        return None
    for destino, claves in reglas.items():
        if any(clave in n for clave in claves):
            return destino
    return None


def mapa_fuente(fuente, reglas, excluir, manual=None):
    """Traduce las clases de un dataset descargado. Respeta el filtro 'solo' de la fuente."""
    nombres = leer_nombres(fuente["dir"])
    manual_fuente = (manual or {}).get(fuente["slug"], {})
    solo = [normalizar(p) for p in fuente.get("solo", [])]
    mapa = {}
    for i, nombre in enumerate(nombres):
        if solo and nombre not in manual_fuente and not any(p in normalizar(nombre) for p in solo):
            mapa[i] = None
        else:
            mapa[i] = mapear_clase(nombre, reglas, excluir, manual_fuente)
    return nombres, mapa


def leer_nombres(dir_dataset):
    """Lee la lista de clases del data.yaml de un dataset descargado."""
    with open(Path(dir_dataset) / "data.yaml", encoding="utf-8") as f:
        datos = yaml.safe_load(f)
    nombres = datos.get("names", [])
    if isinstance(nombres, dict):
        nombres = [nombres[k] for k in sorted(nombres, key=lambda x: int(x))]
    return [str(n) for n in nombres]


def carpeta_split(dir_dataset, split):
    """Roboflow usa 'valid'; algunos datasets usan 'val'."""
    candidatos = [split] + (["val"] if split == "valid" else [])
    for nombre in candidatos:
        ruta = Path(dir_dataset) / nombre / "images"
        if ruta.is_dir():
            return ruta
    return None


def leer_cajas(ruta_txt):
    """Lee etiquetas YOLO. Acepta cajas (cx cy w h) y polígonos (x1 y1 x2 y2 ...).
    Devuelve [(clase, cx, cy, w, h)] recortado a [0, 1]."""
    cajas = []
    ruta_txt = Path(ruta_txt)
    if not ruta_txt.exists():
        return cajas
    for linea in ruta_txt.read_text(encoding="utf-8", errors="ignore").splitlines():
        partes = linea.split()
        if len(partes) < 5:
            continue
        try:
            clase = int(float(partes[0]))
            valores = [float(v) for v in partes[1:]]
        except ValueError:
            continue
        if len(valores) == 4:
            cx, cy, w, h = valores
            x1, y1, x2, y2 = cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
        else:
            if len(valores) % 2:
                valores = valores[:-1]
            xs, ys = valores[0::2], valores[1::2]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        x1, y1 = max(0.0, x1), max(0.0, y1)
        x2, y2 = min(1.0, x2), min(1.0, y2)
        w, h = x2 - x1, y2 - y1
        if w <= 0.002 or h <= 0.002:
            continue
        cajas.append((clase, (x1 + x2) / 2, (y1 + y2) / 2, w, h))
    return cajas


def grupo_base(nombre_archivo):
    """Las copias aumentadas de Roboflow comparten el nombre antes de '.rf.'.
    Se agrupan para que una misma foto no quede en train y en test a la vez."""
    stem = Path(nombre_archivo).stem
    stem = re.split(r"\.rf\.", stem)[0]
    return re.sub(r"_(jpg|jpeg|png|webp|bmp)$", "", stem, flags=re.IGNORECASE)


def valor_hash(texto):
    return int(hashlib.md5(texto.encode("utf-8")).hexdigest()[:8], 16) / 0xFFFFFFFF


def listar_imagenes(carpeta):
    if carpeta is None:
        return []
    return sorted(p for p in carpeta.iterdir() if p.suffix.lower() in EXT_IMAGEN)


def contar_cajas(dir_dataset):
    """Cuenta imágenes y cajas por clase en cada split de un dataset descargado."""
    nombres = leer_nombres(dir_dataset)
    resumen = {}
    for split in SPLITS:
        carpeta = carpeta_split(dir_dataset, split)
        imagenes = listar_imagenes(carpeta)
        conteo = Counter()
        for img in imagenes:
            for clase, *_ in leer_cajas(carpeta.parent / "labels" / f"{img.stem}.txt"):
                nombre = nombres[clase] if clase < len(nombres) else f"clase_{clase}"
                conteo[nombre] += 1
        resumen[split] = {"imagenes": len(imagenes), "cajas": dict(conteo)}
    return resumen


def construir_dataset(nombre, fuentes, reglas, excluir, orden_clases, destino, min_cajas, manual=None):
    """Une varios datasets descargados en uno solo con las clases de RinFix.

    - Las clases que no encajan en las reglas se ignoran.
    - Si una imagen tenía etiquetas pero ninguna se pudo mapear, se descarta (no sabemos qué hay).
    - Si una imagen no tenía etiquetas, se conserva como ejemplo sin daño.
    - Si falta valid o test, se crean a partir de train agrupando las copias aumentadas.
    - Las clases con menos de `min_cajas` cajas de entrenamiento se quitan.
    """
    manual = manual or {}
    destino = Path(destino)
    if destino.exists():
        shutil.rmtree(destino)

    elementos = []
    descartadas = Counter()
    for fuente in fuentes:
        slug = fuente["slug"]
        dir_ds = Path(fuente["dir"])
        _, mapa = mapa_fuente(fuente, reglas, excluir, manual)
        presentes = [s for s in SPLITS if carpeta_split(dir_ds, s) is not None]
        for split in presentes:
            carpeta = carpeta_split(dir_ds, split)
            for img in listar_imagenes(carpeta):
                originales = leer_cajas(carpeta.parent / "labels" / f"{img.stem}.txt")
                mapeadas = [(mapa.get(c), cx, cy, w, h) for c, cx, cy, w, h in originales]
                mapeadas = [m for m in mapeadas if m[0] is not None]
                if originales and not mapeadas:
                    descartadas[slug] += 1
                    continue
                grupo = f"{slug}/{grupo_base(img.name)}"
                split_final = split
                if split == "train":
                    v = valor_hash(grupo)
                    if "test" not in presentes and "valid" not in presentes:
                        split_final = "test" if v < 0.10 else "valid" if v < 0.25 else "train"
                    elif "test" not in presentes:
                        split_final = "test" if v < 0.12 else "train"
                    elif "valid" not in presentes:
                        split_final = "valid" if v < 0.15 else "train"
                elementos.append({"img": img, "split": split_final, "cajas": mapeadas,
                                  "fuente": slug, "grupo": grupo})

    # Garantiza que valid no quede vacío
    if not any(e["split"] == "valid" for e in elementos):
        for e in elementos:
            if e["split"] == "train" and valor_hash(e["grupo"] + "#v") < 0.15:
                e["split"] = "valid"

    conteo_train = Counter(c[0] for e in elementos if e["split"] == "train" for c in e["cajas"])
    clases = [c for c in orden_clases if conteo_train[c] >= min_cajas]
    quitadas = {c: conteo_train[c] for c in orden_clases if c not in clases and conteo_train[c] > 0}
    indice = {c: i for i, c in enumerate(clases)}

    escritas = Counter()
    cajas_por_split = defaultdict(Counter)
    for e in elementos:
        cajas = [c for c in e["cajas"] if c[0] in indice]
        if e["cajas"] and not cajas:
            continue
        carpeta_img = destino / e["split"] / "images"
        carpeta_lbl = destino / e["split"] / "labels"
        carpeta_img.mkdir(parents=True, exist_ok=True)
        carpeta_lbl.mkdir(parents=True, exist_ok=True)
        prefijo = re.sub(r"[^a-zA-Z0-9]+", "-", e["fuente"])
        nuevo = f"{prefijo}__{e['img'].name}"
        shutil.copy2(e["img"], carpeta_img / nuevo)
        lineas = [f"{indice[c]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for c, cx, cy, w, h in cajas]
        (carpeta_lbl / f"{Path(nuevo).stem}.txt").write_text("\n".join(lineas), encoding="utf-8")
        escritas[e["split"]] += 1
        for c in cajas:
            cajas_por_split[e["split"]][c[0]] += 1

    for split in SPLITS:
        (destino / split / "images").mkdir(parents=True, exist_ok=True)
        (destino / split / "labels").mkdir(parents=True, exist_ok=True)

    data_yaml = destino / "data.yaml"
    with open(data_yaml, "w", encoding="utf-8") as f:
        yaml.safe_dump({"path": str(destino), "train": "train/images", "val": "valid/images",
                        "test": "test/images", "names": {i: c for i, c in enumerate(clases)}},
                       f, allow_unicode=True, sort_keys=False)

    print(f"\n=== Dataset '{nombre}' ===")
    print("Clases finales:", clases if clases else "NINGUNA (revisa el mapeo)")
    for split in SPLITS:
        print(f"  {split:5s}: {escritas[split]:5d} imágenes | cajas: {dict(cajas_por_split[split])}")
    if quitadas:
        print(f"  Clases quitadas por tener menos de {min_cajas} cajas en train: {quitadas}")
    if descartadas:
        print(f"  Imágenes descartadas porque ninguna etiqueta encajó: {dict(descartadas)}")
    return {"data_yaml": data_yaml, "clases": clases, "imagenes": dict(escritas),
            "cajas": {s: dict(cajas_por_split[s]) for s in SPLITS}, "quitadas": quitadas}


def dibujar(ruta_img, cajas, nombres):
    """Dibuja cajas normalizadas sobre una imagen."""
    img = Image.open(ruta_img).convert("RGB")
    ancho, alto = img.size
    dibujo = ImageDraw.Draw(img)
    grosor = max(2, ancho // 200)
    for clase, cx, cy, w, h in cajas:
        color = COLORES[clase % len(COLORES)]
        x1, y1 = (cx - w / 2) * ancho, (cy - h / 2) * alto
        x2, y2 = (cx + w / 2) * ancho, (cy + h / 2) * alto
        dibujo.rectangle([x1, y1, x2, y2], outline=color, width=grosor)
        texto = nombres[clase] if clase < len(nombres) else str(clase)
        dibujo.text((x1 + 3, max(0, y1 - 12)), texto, fill=color)
    return img


def mostrar_muestras(dir_dataset, split="train", n=6, titulo="", semilla=7):
    """Muestra n imágenes con sus etiquetas para revisar a ojo."""
    nombres = leer_nombres(dir_dataset)
    carpeta = carpeta_split(dir_dataset, split)
    imagenes = listar_imagenes(carpeta)
    if not imagenes:
        print(f"Sin imágenes en {split} para {titulo}")
        return
    random.Random(semilla).shuffle(imagenes)
    imagenes = imagenes[:n]
    columnas = min(3, len(imagenes))
    filas = (len(imagenes) + columnas - 1) // columnas
    fig, ejes = plt.subplots(filas, columnas, figsize=(5 * columnas, 5 * filas))
    ejes = ejes.flatten() if hasattr(ejes, "flatten") else [ejes]
    for eje, img in zip(ejes, imagenes):
        cajas = leer_cajas(carpeta.parent / "labels" / f"{img.stem}.txt")
        eje.imshow(dibujar(img, cajas, nombres))
        eje.set_title(img.name[:40], fontsize=8)
        eje.axis("off")
    for eje in ejes[len(imagenes):]:
        eje.axis("off")
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()


def convertir_coco_a_yolo(dir_dataset):
    """Convierte una descarga de Roboflow en formato COCO (por ejemplo, de un proyecto de
    segmentación) a la estructura YOLO split/images + split/labels con cajas."""
    dir_dataset = Path(dir_dataset)
    categorias = {}
    por_split = {}
    for carpeta in sorted(d for d in dir_dataset.iterdir() if d.is_dir()):
        archivo = carpeta / "_annotations.coco.json"
        if archivo.exists():
            datos = json.loads(archivo.read_text(encoding="utf-8"))
            for categoria in datos.get("categories", []):
                categorias[categoria["id"]] = categoria["name"]
            por_split[carpeta.name] = datos
    if not por_split:
        raise RuntimeError("No se encontró ningún _annotations.coco.json en la descarga")

    ids = sorted(categorias)
    indice = {cid: i for i, cid in enumerate(ids)}
    for split, datos in por_split.items():
        carpeta = dir_dataset / split
        (carpeta / "images").mkdir(exist_ok=True)
        (carpeta / "labels").mkdir(exist_ok=True)
        anotaciones = defaultdict(list)
        for anotacion in datos.get("annotations", []):
            anotaciones[anotacion["image_id"]].append(anotacion)
        for imagen in datos.get("images", []):
            origen = carpeta / imagen["file_name"]
            if not origen.exists():
                continue
            destino = carpeta / "images" / Path(imagen["file_name"]).name
            shutil.move(str(origen), destino)
            ancho, alto = imagen["width"], imagen["height"]
            lineas = []
            for anotacion in anotaciones.get(imagen["id"], []):
                x, y, w, h = anotacion.get("bbox") or [0, 0, 0, 0]
                if (w <= 0 or h <= 0) and isinstance(anotacion.get("segmentation"), list):
                    puntos = [v for poligono in anotacion["segmentation"] if isinstance(poligono, list)
                              for v in poligono]
                    xs, ys = puntos[0::2], puntos[1::2]
                    if xs and ys:
                        x, y, w, h = min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)
                if w <= 0 or h <= 0 or anotacion["category_id"] not in indice:
                    continue
                lineas.append(f"{indice[anotacion['category_id']]} {(x + w / 2) / ancho:.6f} "
                              f"{(y + h / 2) / alto:.6f} {w / ancho:.6f} {h / alto:.6f}")
            (carpeta / "labels" / f"{destino.stem}.txt").write_text("\n".join(lineas), encoding="utf-8")

    with open(dir_dataset / "data.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump({"names": [categorias[c] for c in ids], "nc": len(ids)}, f, allow_unicode=True)
    return dir_dataset


def ultima_version(proyecto):
    """Busca el número de versión más alto publicado de un proyecto de Roboflow."""
    try:
        numeros = []
        for v in proyecto.versions():
            num = str(getattr(v, "version", "")).split("/")[-1]
            if num.isdigit():
                numeros.append(int(num))
        if numeros:
            return max(numeros)
    except Exception as error:
        print("   No se pudo listar versiones:", error)
    for numero in range(15, 0, -1):
        try:
            proyecto.version(numero)
            return numero
        except Exception:
            continue
    return None


print("Funciones listas.")

## 4. Descargar los datasets

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
DESCARGADAS = []
for fuente in FUENTES:
    slug = f"{fuente['workspace']}/{fuente['proyecto']}"
    destino = RAW_DIR / fuente["proyecto"]
    print(f"\nDescargando {slug} ...")
    try:
        proyecto = rf.workspace(fuente["workspace"]).project(fuente["proyecto"])
        version = fuente.get("version") or ultima_version(proyecto)
        if version is None:
            raise RuntimeError("no tiene versiones publicadas")
        try:
            dataset = proyecto.version(version).download("yolov8", location=str(destino), overwrite=True)
            dir_ds = Path(dataset.location)
        except Exception as error:
            if "invalid format" not in str(error).lower():
                raise
            print("   No es un proyecto de detección: se descarga en COCO y se convierte a cajas.")
            try:
                dataset = proyecto.version(version).download("coco-segmentation", location=str(destino),
                                                             overwrite=True)
            except Exception:
                dataset = proyecto.version(version).download("coco", location=str(destino), overwrite=True)
            dir_ds = convertir_coco_a_yolo(dataset.location)
        DESCARGADAS.append({**fuente, "slug": slug, "dir": dir_ds, "version_usada": version})
        print(f"   OK: versión {version}")
    except Exception as error:
        print(f"   ERROR: no se pudo descargar ({error})")

print(f"\nDescargados {len(DESCARGADAS)} de {len(FUENTES)} datasets.")

## 5. Revisar qué trae cada dataset

Para cada dataset se muestran sus clases, cómo se traducen a las clases de RinFix y algunas fotos con sus etiquetas.

Mira las fotos: si un dataset no es de rines de carro (por ejemplo, ruedas de tren o fotos de fábrica), cambia su `incluir` a `False` en la celda de fuentes y vuelve a ejecutar desde ahí.

In [ ]:
for fuente in DESCARGADAS:
    es_rin = fuente["tipo"] == "rin"
    reglas = REGLAS_RIN if es_rin else REGLAS_DANOS
    excluir = EXCLUIR_RIN if es_rin else EXCLUIR_DANOS
    nombres, mapa = mapa_fuente(fuente, reglas, excluir, MAPEO_MANUAL)
    estado = "SE USA" if fuente["incluir"] else "NO SE USA (incluir=False)"
    print("=" * 80)
    print(f"{fuente['slug']}  (v{fuente['version_usada']}, tipo {fuente['tipo']}, {estado})")
    print("Nota:", fuente["nota"])
    print("Clases originales -> RinFix:")
    for i, nombre in enumerate(nombres):
        print(f"   {nombre!r:30} -> {mapa[i] or 'ignorada'}")
    for split, datos in contar_cajas(fuente["dir"]).items():
        print(f"   {split:5s}: {datos['imagenes']:5d} imágenes | {datos['cajas']}")
    mostrar_muestras(fuente["dir"], "train", n=6, titulo=fuente["slug"])

## 6. Unir los datasets

In [ ]:
usar_rin = [f for f in DESCARGADAS if f["tipo"] == "rin" and f["incluir"]]
usar_danos = [f for f in DESCARGADAS if f["tipo"] == "danos" and f["incluir"]]

DS_RIN = construir_dataset("rin", usar_rin, REGLAS_RIN, EXCLUIR_RIN, ["rin"],
                           DATA_DIR / "rin", MIN_CAJAS, MAPEO_MANUAL)
DS_DANOS = construir_dataset("danos", usar_danos, REGLAS_DANOS, EXCLUIR_DANOS, ORDEN_DANOS,
                             DATA_DIR / "danos", MIN_CAJAS, MAPEO_MANUAL)

assert DS_RIN["clases"], "El dataset de rin quedó sin clases: revisa las fuentes y el mapeo."
assert DS_DANOS["clases"], "El dataset de daños quedó sin clases: revisa las fuentes y el mapeo."

In [ ]:
mostrar_muestras(DATA_DIR / "rin", "train", n=6, titulo="Dataset unido: rin")
mostrar_muestras(DATA_DIR / "danos", "train", n=9, titulo="Dataset unido: daños")

## 7. Entrenar el modelo de rin

In [ ]:
from ultralytics import YOLO

modelo_rin = YOLO(MODELO_BASE_RIN)
modelo_rin.train(
    data=str(DS_RIN["data_yaml"]),
    epochs=EPOCAS_RIN,
    patience=PACIENCIA_RIN,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(RUNS_DIR),
    name="rin",
    exist_ok=True,
    seed=42,
    plots=True,
)
BEST_RIN = Path(modelo_rin.trainer.save_dir) / "weights" / "best.pt"
print("Mejor modelo de rin:", BEST_RIN)

## 8. Entrenar el modelo de daños

In [ ]:
modelo_danos = YOLO(MODELO_BASE_DANOS)
modelo_danos.train(
    data=str(DS_DANOS["data_yaml"]),
    epochs=EPOCAS_DANOS,
    patience=PACIENCIA_DANOS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(RUNS_DIR),
    name="danos",
    exist_ok=True,
    seed=42,
    plots=True,
)
BEST_DANOS = Path(modelo_danos.trainer.save_dir) / "weights" / "best.pt"
print("Mejor modelo de daños:", BEST_DANOS)

## 9. Medir en el conjunto de prueba

El conjunto de prueba no se usó para entrenar. Cómo leer los números:

- **Precisión:** de lo que el modelo marcó, cuánto era correcto.
- **Recall:** de los daños reales, cuántos encontró. Para RinFix importa más no dejar pasar daños.
- **mAP50:** resumen general (de 0 a 1). Por debajo de 0,5 el modelo todavía no es confiable.

In [ ]:
def evaluar(best, info, nombre):
    modelo = YOLO(str(best))
    tiene_test = info["imagenes"].get("test", 0) > 0
    split = "test" if tiene_test else "val"
    metricas = modelo.val(data=str(info["data_yaml"]), split=split, imgsz=IMGSZ, plots=True,
                          project=str(RUNS_DIR), name=f"{nombre}_eval", exist_ok=True)
    por_clase = []
    for i, indice in enumerate(metricas.box.ap_class_index):
        p, r, ap50, ap = metricas.box.class_result(i)
        por_clase.append({"clase": modelo.names[int(indice)], "precision": round(float(p), 3),
                          "recall": round(float(r), 3), "mAP50": round(float(ap50), 3),
                          "mAP50_95": round(float(ap), 3)})
    resumen = {
        "modelo": nombre, "split": split,
        "precision": round(float(metricas.box.mp), 3), "recall": round(float(metricas.box.mr), 3),
        "mAP50": round(float(metricas.box.map50), 3), "mAP50_95": round(float(metricas.box.map), 3),
        "por_clase": por_clase, "imagenes": info["imagenes"], "cajas": info["cajas"],
        "dir_eval": str(metricas.save_dir),
    }
    print(f"\n=== {nombre} ({split}) ===")
    print(f"precisión {resumen['precision']} | recall {resumen['recall']} | "
          f"mAP50 {resumen['mAP50']} | mAP50-95 {resumen['mAP50_95']}")
    for fila in por_clase:
        print(f"   {fila['clase']:10s} precisión {fila['precision']:.3f}  recall {fila['recall']:.3f}  "
              f"mAP50 {fila['mAP50']:.3f}")
    return resumen


METRICAS = {"rin": evaluar(BEST_RIN, DS_RIN, "rin"), "danos": evaluar(BEST_DANOS, DS_DANOS, "danos")}

## 10. Exportar a LiteRT y guardar en Drive

Desde Ultralytics 8.4.83 el formato para Android es LiteRT (`format="litert"`), que genera un archivo `.tflite`. Se guardan dos versiones de cada modelo: precisión completa (`fp32`) y una más liviana (`w8a32`) para comparar velocidad en el teléfono.

In [ ]:
def exportar(best, nombre, carpeta):
    carpeta.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best, carpeta / f"{nombre}.pt")
    try:
        ruta = YOLO(str(best)).export(format="litert", imgsz=IMGSZ)
    except Exception as error:
        print("LiteRT no disponible en esta versión, se usa tflite:", error)
        ruta = YOLO(str(best)).export(format="tflite", imgsz=IMGSZ)
    shutil.copy2(ruta, carpeta / f"{nombre}_fp32.tflite")
    try:
        ruta_q = YOLO(str(best)).export(format="litert", imgsz=IMGSZ, quantize="w8a32")
        shutil.copy2(ruta_q, carpeta / f"{nombre}_w8a32.tflite")
    except Exception as error:
        print("No se pudo crear la versión w8a32:", error)
    nombres = YOLO(str(best)).names
    (carpeta / f"{nombre}_labels.txt").write_text(
        "\n".join(nombres[i] for i in sorted(nombres)), encoding="utf-8")


for nombre, best in (("rin", BEST_RIN), ("danos", BEST_DANOS)):
    destino = SALIDA / nombre
    exportar(best, nombre, destino)
    entrenamiento = RUNS_DIR / nombre
    for archivo in ("results.png", "results.csv", "args.yaml"):
        if (entrenamiento / archivo).exists():
            shutil.copy2(entrenamiento / archivo, destino / archivo)
    dir_eval = Path(METRICAS[nombre]["dir_eval"])
    for grafica in dir_eval.glob("*.png"):
        shutil.copy2(grafica, destino / f"prueba_{grafica.name}")
    (destino / "metricas.json").write_text(json.dumps(METRICAS[nombre], indent=2, ensure_ascii=False),
                                          encoding="utf-8")

creditos = ["# Créditos de los datos", "",
            "Modelos entrenados con estos datasets de Roboflow Universe (licencia CC BY 4.0):", ""]
for fuente in DESCARGADAS:
    if fuente["incluir"]:
        creditos.append(f"- {fuente['slug']} v{fuente['version_usada']}: "
                        f"https://universe.roboflow.com/{fuente['slug']}")
creditos += ["", "Detector: Ultralytics YOLOv8 (AGPL-3.0)."]
(SALIDA / "CREDITOS.md").write_text("\n".join(creditos), encoding="utf-8")

print("Listo. Archivos guardados en:", SALIDA)
for archivo in sorted(SALIDA.rglob("*")):
    if archivo.is_file():
        print(f"   {archivo.relative_to(SALIDA)}  ({archivo.stat().st_size / 1e6:.1f} MB)")

## 11. Probar con tus propias fotos (opcional)

Sube fotos de rines tomadas con el celular. El cuaderno busca el rin, lo recorta y busca daños dentro del recorte, igual que lo hará la app. Los números junto a cada caja son el puntaje interno del modelo: sirven para calibrar, no se muestran al cliente.

In [ ]:
from google.colab import files

CONF = 0.25
subidas = files.upload()
detector_rin = YOLO(str(BEST_RIN))
detector_danos = YOLO(str(BEST_DANOS))

for nombre_archivo in subidas:
    imagen = Image.open(nombre_archivo).convert("RGB")
    ancho, alto = imagen.size
    resultado_rin = detector_rin.predict(imagen, imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    dibujo_img = imagen.copy()
    dibujo = ImageDraw.Draw(dibujo_img)
    grosor = max(3, ancho // 150)

    if len(resultado_rin.boxes):
        mejor = int(resultado_rin.boxes.conf.argmax())
        x1, y1, x2, y2 = resultado_rin.boxes.xyxy[mejor].tolist()
        margen_x, margen_y = (x2 - x1) * 0.08, (y2 - y1) * 0.08
        x1, y1 = max(0, x1 - margen_x), max(0, y1 - margen_y)
        x2, y2 = min(ancho, x2 + margen_x), min(alto, y2 + margen_y)
        dibujo.rectangle([x1, y1, x2, y2], outline="#2E9E5B", width=grosor)
        dibujo.text((x1 + 5, y1 + 5), f"rin {float(resultado_rin.boxes.conf[mejor]):.2f}", fill="#2E9E5B")
        recorte = imagen.crop((int(x1), int(y1), int(x2), int(y2)))
        desplazamiento = (x1, y1)
    else:
        print(f"{nombre_archivo}: no se encontró un rin; se buscan daños en la foto completa.")
        recorte = imagen
        desplazamiento = (0, 0)

    resultado_danos = detector_danos.predict(recorte, imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    hallazgos = []
    for caja, clase, puntaje in zip(resultado_danos.boxes.xyxy.tolist(),
                                    resultado_danos.boxes.cls.tolist(),
                                    resultado_danos.boxes.conf.tolist()):
        dx, dy = desplazamiento
        bx1, by1, bx2, by2 = caja[0] + dx, caja[1] + dy, caja[2] + dx, caja[3] + dy
        etiqueta = detector_danos.names[int(clase)]
        color = COLORES[(int(clase) + 1) % len(COLORES)]
        dibujo.rectangle([bx1, by1, bx2, by2], outline=color, width=grosor)
        dibujo.text((bx1 + 5, by1 + 5), f"{etiqueta} {puntaje:.2f}", fill=color)
        hallazgos.append(f"{etiqueta} ({puntaje:.2f})")

    plt.figure(figsize=(9, 9))
    plt.imshow(dibujo_img)
    plt.axis("off")
    plt.title(f"{nombre_archivo}: {', '.join(hallazgos) if hallazgos else 'sin daños detectados'}")
    plt.show()

## 12. Qué sigue

1. Abre en Google Drive la carpeta `RinFix/ml/modelos/<fecha>` y descárgala.
2. Pásale a Claude el contenido de `rin/metricas.json` y `danos/metricas.json`, y las fotos de prueba si las hiciste.
3. Con esos números decidimos si el modelo entra a la app (fase 6) o si hace falta más datos.

Los archivos `.pt` y `.tflite` no se suben a GitHub desde la carpeta `ml/`: están en el `.gitignore`. Cuando el modelo se integre a la app, se copiará el `.tflite` elegido a `app/src/main/assets/`.